# Fine-tune:

End-to-end encoder fine-tuning for `binary_reliability` (`binary_relevancy AND binary_faithfulness`).

Pipeline: CSV preprocessing -> tokenization -> encoder + classification head -> test metrics.

## Config

In [1]:
from pathlib import Path

DATA_PATH = Path("data.csv")
MODEL_NAME = "deepvk/RuModernBERT-base"

TARGET = "binary_reliability"
CHUNK_COLS = [f"chunk_{i}" for i in range(1, 9)]

SEED = 42
TEST_SIZE = 0.2
MAX_LENGTH = 8096
BATCH_SIZE = 8
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "checkpoints"

In [2]:
import os
import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [5]:
df.iloc[0]['full_dialog']

'Ассистент: Приветствую, [NAME]! На связи Альфа-Помощник 🦸\u200d♂️\nУ вас старая версия приложения, поэтому некоторые функции могут не работать.\n\n🔄 Обновите или скачайте приложение на андроид [на сайте]([URL] что вы не выбрали категории кэшбэка на август.\n\n👉 Можете [выбрать категории сейчас.]([URL]\nКлиент: Установка пей платеж'

In [6]:
df.iloc[20]['full_dialog']

'Ассистент: Приветствую, Лейсан! На связи Альфа-Помощник 🦸\u200d♂️\nУ вас старая версия приложения, поэтому некоторые функции могут не работать.\n\n🔄 Обновите или скачайте приложение на андроид [на сайте]([URL] что вы не выбрали категории кэшбэка на август.\n\n👉 Можете [выбрать категории сейчас.]([URL]\nКлиент: Добрый день. Пришло смс:\nНеобходимо проверить реквизиты получателя: банк, ИНН и номер счёта. Пожалуйста, свяжитесь с нами по телефону, указанному на обратной стороне карты, чтобы подтвердить платёж\n\nЧто нужно сделать?'

## Preprocessing

In [7]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

def clean_text(value) -> str:
    return "" if pd.isna(value) else str(value).strip()


def build_input(row) -> str:
    chunks = "\n\n".join(t for t in map(clean_text, (row[c] for c in CHUNK_COLS)) if t)
    return (
        f"query: {clean_text(row['full_query'])}\n"
        f"answer: {clean_text(row['answer'])}\n"
        f"context: {chunks}"
    )

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[TARGET],
)
train_df.shape, test_df.shape

((3825, 15), (957, 15))

In [ ]:

df["text"] = df.apply(build_input, axis=1)
df[TARGET] = (df["binary_relevancy"].astype(bool) & df["binary_faithfulness"].astype(bool)).astype(int)
df[["text", TARGET]].head()

## Data pipeline

In [12]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)


def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["text", TARGET]].rename(columns={TARGET: "labels"}), preserve_index=False)
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True)
    return ds.remove_columns("text")


train_ds = to_dataset(train_df)
test_ds = to_dataset(test_df)
collator = DataCollatorWithPadding(tokenizer)

Map: 100%|██████████| 957/957 [00:01<00:00, 828.18 examples/s]


## Model

In [ ]:
import torch.nn as nn
from transformers import AutoModel


def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


class Classifier(nn.Module):
    def __init__(self, name, pos_weight):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, trust_remote_code=True)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.encoder.config.hidden_size, 1))
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.head(mean_pool(out.last_hidden_state, attention_mask)).squeeze(-1)
        loss = self.loss_fn(logits, labels.float()) if labels is not None else None
        return {"loss": loss, "logits": logits}


pos = train_df[TARGET].sum()
pos_weight = torch.tensor([(len(train_df) - pos) / max(pos, 1)])
model = Classifier(MODEL_NAME, pos_weight)

## Training

In [15]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy='epoch',
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.410485,0.430148
2,0.398333,0.407655


TrainOutput(global_step=958, training_loss=0.4150325547678237, metrics={'train_runtime': 1240.8801, 'train_samples_per_second': 6.165, 'train_steps_per_second': 0.772, 'total_flos': 0.0, 'train_loss': 0.4150325547678237, 'epoch': 2.0})

## Test metrics

In [16]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

logits = trainer.predict(test_ds).predictions
y_prob = 1 / (1 + np.exp(-logits))
y_true = np.array(test_ds["labels"])
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan"),
}
pd.Series(metrics)

accuracy     0.641588
precision    0.800781
recall       0.629800
f1           0.705073
roc_auc      0.692720
dtype: float64

In [17]:
print(classification_report(y_true, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.46      0.67      0.54       306
           1       0.80      0.63      0.71       651

    accuracy                           0.64       957
   macro avg       0.63      0.65      0.62       957
weighted avg       0.69      0.64      0.65       957

